In [ ]:
import h5py
import numpy as np
import torch

In [2]:
with h5py.File(r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5", "r") as f:
    print("Keys:", list(f.keys()))

    seqs = f["sequence_integer"][34144:34176]
    charges_oh = f["precursor_charge_onehot"][34144:34176]
    intensities = f["intensities_raw"][34144:34176]
    masses_raw = f["masses_raw"][:1]
    masses_pred = f["masses_pred"][:1]
    rev = f["reverse"][:1]
    rawfile = f["rawfile"][:1]
    collision_energy = f["collision_energy_aligned_normed"][:10]
    seq_ohs = f["sequence_onehot"][:25]

Keys: ['collision_energy', 'collision_energy_aligned', 'collision_energy_aligned_normed', 'intensities_raw', 'masses_pred', 'masses_raw', 'method', 'precursor_charge_onehot', 'rawfile', 'reverse', 'scan_number', 'score', 'sequence_integer', 'sequence_onehot']


In [ ]:
def get_x0(x_1: torch.Tensor, max_scale=False):
    # Sample x_0 from a distribution and scale based on the max value of itself
    # x_1 shape: (batch_size, length, 6)
    x_0 = torch.randn_like(x_1)
    if max_scale:
        if x_1.dim() == 1:
            max_val = x_0.abs().max(keepdim=True)
            max_val  = torch.clamp(max_val, min=1e-8)
            x_0 = x_0 / max_val
        elif x_1.dim() == 2:  # (B, D)
            max_val = x_0.abs().max(dim=1, keepdim=True)[0]

            max_val = torch.clamp(max_val, min=1e-8)
            x_0 = x_0 / max_val

        elif x_1.dim() == 3:  # (B, L, D)
            max_val = x_0.abs().amax(dim=(1, 2), keepdim=True)
            max_val = torch.clamp(max_val, min=1e-8)
            x_0 = x_0 / max_val
    return x_0


In [13]:
intensities[0].shape

(174,)

In [43]:
test = torch.tensor([[1, 2, 3, 10, -5], [3,4, 10, 9, -3]], dtype=torch.float32)

In [44]:
max_val = torch.clamp(test.abs().max(dim=1, keepdim=True)[0], min=1e-8)

In [45]:
test  / max_val

tensor([[ 0.1000,  0.2000,  0.3000,  1.0000, -0.5000],
        [ 0.3000,  0.4000,  1.0000,  0.9000, -0.3000]])

In [48]:
intensities.shape

(32, 174)

In [69]:
x = get_x0(torch.tensor(intensities), max_scale=True)

In [71]:
x[0]

tensor([-0.1886, -0.1817, -0.1561, -0.0949, -0.0445,  0.2253, -0.0608,  0.2209,
        -0.5046, -0.2508, -0.1030,  0.2926, -0.1080, -0.1062, -0.0278,  0.9433,
        -0.5276,  0.2055, -0.0180,  0.0743, -0.3384, -0.3971, -0.1558,  0.0067,
         0.4367,  0.2902, -0.4856,  0.1394, -0.6311, -0.3493,  0.3547, -0.7995,
         0.0437,  0.2926, -0.3217,  0.2253, -0.3657,  0.7616,  0.3979, -0.2775,
         0.2108,  0.3111,  0.2355,  0.2998, -0.0465,  0.4242,  0.4465, -0.0558,
         0.5158, -0.0797, -0.1199, -0.0721,  0.5449,  0.5089, -0.2627, -0.7600,
        -0.0382,  0.2764,  0.3363,  0.1957,  0.5168,  0.4571,  0.0762,  0.3111,
        -0.2693,  0.4198,  0.6645, -0.0502, -0.3709, -0.5967, -0.0327,  0.1915,
        -0.2827,  0.1755,  0.4921, -0.7173, -0.5262, -0.5376, -0.2192,  0.1967,
         0.1574, -0.5128,  0.1840, -0.0157, -0.0809,  0.1983, -0.3693,  0.2687,
         0.3969,  0.3543, -0.1855,  0.4427, -0.5646,  0.0403, -0.0088,  0.2573,
        -0.5081, -0.0592,  0.2343,  0.46

In [70]:
x[1]

tensor([-0.5974,  0.0018, -0.2453,  0.3505, -0.1519, -0.3054, -0.3737,  0.1522,
        -0.1303,  0.2916,  0.2607,  0.3318,  0.0080,  0.5621, -0.4785,  0.1630,
         0.2046,  0.0281, -0.4678, -0.4607, -0.5659, -0.3132,  0.1218,  0.1328,
        -0.3067,  0.1297, -0.0433,  0.2356, -0.5294, -0.0747,  0.1986, -0.6306,
        -0.4598, -0.4102,  1.0000, -0.0685,  0.0297,  0.7205, -0.5334,  0.1788,
         0.2975, -0.0739, -0.5129,  0.0196,  0.2218,  0.1938, -0.3830, -0.0330,
         0.6234, -0.7848, -0.1587,  0.1480, -0.7934, -0.0228, -0.5393, -0.1012,
         0.4437,  0.5148, -0.2975,  0.1204,  0.2051,  0.0259,  0.0187,  0.4565,
         0.5386, -0.2638,  0.1626, -0.3635, -0.4520, -0.2933,  0.2047, -0.6602,
         0.1465, -0.3520,  0.7697, -0.3928, -0.3267, -0.1479, -0.2261,  0.3609,
        -0.0332, -0.3603, -0.0837, -0.2853, -0.4009, -0.4465, -0.5158, -0.5804,
         0.2380,  0.0947,  0.4860, -0.0511,  0.4072,  0.5528, -0.1861, -0.0231,
        -0.1405, -0.1939, -0.1029,  0.31

In [ ]:
charges = np.argmax(charges_oh, axis = 1)

In [ ]:
embedding = torch.nn.Embedding(10, 3)

In [ ]:
embedding(torch.tensor(10))

IndexError: index out of range in self

In [ ]:
charges[0] = 6

In [ ]:
charges

array([6, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 1, 1, 1, 3, 0, 1, 2, 2, 1, 2, 2,
       3, 2, 2, 3, 1, 3, 1, 1, 1, 2])

In [ ]:
np.min(np.argmax(charges_oh, axis=1))

np.int64(0)

In [ ]:
from models.embedding import ChargeEmbedding

In [ ]:
charg_embedding = ChargeEmbedding(1, 6, use_layernorm=False)

In [ ]:
charg_embedding(torch.tensor(charges[0:15], dtype=torch.long).unsqueeze(1))

Charge shape: torch.Size([15, 1])
tensor([[6],
        [1],
        [1],
        [1],
        [2],
        [1],
        [1],
        [1],
        [2],
        [2],
        [1],
        [1],
        [1],
        [1],
        [3]])
Emb shape: torch.Size([15, 3])
Scalar shape: torch.Size([15, 1])


tensor([[ 1.0000, -0.1618, -0.5278,  0.1490],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.2000,  0.4989, -0.7844, -1.1747],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.2000,  0.4989, -0.7844, -1.1747],
        [ 0.2000,  0.4989, -0.7844, -1.1747],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.0000, -0.4474, -1.8566, -0.9292],
        [ 0.4000, -2.0290, -0.2719,  0.9047]], grad_fn=<CatBackward0>)

In [ ]:
np.max(seqs)

np.int64(21)